# 01 · Hello Flyte

This is the smallest Flyte program that still shows the whole shape of the system — a `TaskEnvironment`,
a couple of `@env.task` functions, and a workflow that is just a task calling
other tasks. [02](./02_core_pipeline.ipynb) and
[03](./03_production_pipeline.ipynb) are this same shape, scaled up.

**Flyte features**

1. `TaskEnvironment` — groups image, resources, and config for a set of tasks
2. `@env.task` — turns a Python function into a containerized, tracked task
3. **sync vs async tasks** — the same decorator over `def` and `async def`
4. **workflow = task calling tasks** — composition is plain Python
5. `flyte.init_from_config` / `flyte.run` — connect and run locally vs remotely
6. **runs & actions** — one `flyte.run` is a *run*; each task call is an *action*

Docs: [tasks](https://www.union.ai/docs/v2/union/user-guide/core-concepts/tasks/) ·
[task environments](https://www.union.ai/docs/v2/union/user-guide/core-concepts/task-environment/) ·
[running locally](https://www.union.ai/docs/v2/union/user-guide/run-modes/running-locally/) ·
[runs and actions](https://www.union.ai/docs/v2/union/user-guide/core-concepts/runs-and-actions/)

### Connect to a deployment

`flyte.init_from_config` reads a config file and opens the connection
(endpoint, org, project, domain) every later `flyte.run` targets. Nothing is
submitted yet. Point it at a different `config.yaml` to run the same code on
another cluster.

In [6]:
from pathlib import Path

import flyte

# Points at this workshop's .flyte/config.yaml. Swap in your cluster's endpoint/org/project there.
flyte.init_from_config(Path(".flyte") / "config.yaml")

## 1. `TaskEnvironment`

Bundles what a group of tasks needs to run — roughly one env ↔ one pod spec:

- **`name`** — prefixes pod names and groups tasks in the run UI
- **`image`** — the container image (defaults to the Flyte base; custom in 02)
- **`resources`** — CPU / memory / GPU / disk per task pod

Tasks that share sizing go in one env; a light coordinator and a heavy worker
go in separate envs ([02](./02_core_pipeline.ipynb)). Here everything is tiny,
so one env is enough.

In [ ]:
env = flyte.TaskEnvironment(
    name="first_task",
    resources=flyte.Resources(cpu=1, memory="250Mi"),
)

## 2. `@env.task` — sync and async

`@env.task` promotes a function to a task: on the cluster each call is packaged,
scheduled onto its own pod, run, and tracked as an **action**. Type hints drive
serialization between tasks. Same decorator over both styles:

- **sync** (`join_names`) — a simple leaf; call it directly
- **async** (`get_name_length`) — enables `asyncio.gather` / `flyte.map`
  fan-out and reusable-container concurrency. `await` it, or call `.aio` from
  other async code.

In [ ]:
# Sync task — a plain Python function.
@env.task
def join_names(first_name: str, last_name: str, config: dict) -> str:
    return f"{first_name} {last_name}"


# Async task — same decorator, async def. Use this when you want to await
# other tasks or run I/O concurrently (chapter 02).
@env.task
async def get_name_length(name: str) -> int:
    return len(name)

## 3. A workflow *is* a task that calls tasks

A workflow is just an `@env.task` that calls other tasks, so orchestration is
plain Python. Each call becomes its **own action**,
and return values pass automatically (Flyte offloads them to object storage and
reloads them). Sync tasks are called directly; async tasks are `await`ed.

In [ ]:
@env.task
async def main(first_name: str = "Ada", last_name: str = "Lovelace") -> str:
    print("Starting first workflow...")

    full_name = join_names(first_name, last_name, {"01": "value"})  # sync task call
    name_length = await get_name_length(full_name)  # async task call

    result = f"'{full_name}' has {name_length} characters"
    print(result)
    return result

## 4. The dev loop: local vs remote

Same code, two ways to run:

- **Local** — `flyte run --local 01_hello_flyte.py main` runs the bodies
  in-process: no cluster, no tracking. Fast inner loop.
- **Remote** — `flyte.run(main)` (below) ships the code and runs each call as a
  tracked container.

`flyte.run` returns a handle: `run.url` (run page), `run.wait()` (block until
done), `run.outputs()` (results). One run, one action per task call.

In [ ]:
run = flyte.run(main)
print(f"Run URL: {run.url}")
run.wait()
run.outputs()

## Further reading

- Union docs: [core concepts](https://www.union.ai/docs/v2/union/user-guide/core-concepts/tasks/)
- Next: [02_core_pipeline](./02_core_pipeline.ipynb) — a real text-processing
  pipeline with fan-out, files, and multiple environments